# Kaggle 02. Standalone Phase 1 Encoder Baselines + Analytics

Этот ноутбук самодостаточен. Он ожидает, что в `/kaggle/input` лежит датасет, в котором есть и `cocolofa_ru_v2.jsonl`, и `cocolofa_ru_v2_masked.jsonl`. Ноутбук обучает four baseline-конфигурации, строит сводные метрики и добавляет исследовательскую аналитику: длины текстов, частотные слова по классам, ROC-AUC, проекцию эмбеддингов и attention visualization.

In [ ]:
%pip install -q transformers sentencepiece scikit-learn seaborn plotly bertviz

In [ ]:
import json
import logging
import random
import re
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from statistics import mean, stdev
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import torch
from bertviz import head_view
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('kaggle_phase1')
sns.set_theme(style='whitegrid')


In [ ]:
# Автоматически находим raw и masked JSONL в /kaggle/input.
INPUT_JSONL_CANDIDATES = sorted(Path('/kaggle/input').rglob('*.jsonl'))
assert INPUT_JSONL_CANDIDATES, 'Upload the dataset JSONL files to /kaggle/input first.'

for path in INPUT_JSONL_CANDIDATES:
    print(path)

RAW_DATASET_PATH = next((path for path in INPUT_JSONL_CANDIDATES if path.name == 'cocolofa_ru_v2.jsonl'), None)
MASKED_DATASET_PATH = next((path for path in INPUT_JSONL_CANDIDATES if path.name == 'cocolofa_ru_v2_masked.jsonl'), None)

assert RAW_DATASET_PATH is not None, 'cocolofa_ru_v2.jsonl is missing in /kaggle/input.'
assert MASKED_DATASET_PATH is not None, 'cocolofa_ru_v2_masked.jsonl is missing in /kaggle/input.'

OUTPUT_ROOT = Path('/kaggle/working/phase1_baselines')
SEEDS = [42, 52, 62]
MAX_RUNS = None  # Set to 1 or 2 only for debugging.
RUN_CONTEXT_WINDOW_SWEEP = False

{
    'raw_dataset_path': str(RAW_DATASET_PATH),
    'masked_dataset_path': str(MASKED_DATASET_PATH),
    'output_root': str(OUTPUT_ROOT),
    'seeds': SEEDS,
    'max_runs': MAX_RUNS,
    'run_context_window_sweep': RUN_CONTEXT_WINDOW_SWEEP,
}


In [ ]:
LABEL_TO_ID = {
    'none': 0,
    'appeal to authority': 1,
    'appeal to majority': 2,
    'appeal to nature': 3,
    'appeal to tradition': 4,
    'appeal to worse problems': 5,
    'false dilemma': 6,
    'hasty generalization': 7,
    'slippery slope': 8,
}
ID_TO_LABEL = {value: key for key, value in LABEL_TO_ID.items()}
N_CLASSES = len(LABEL_TO_ID)
ACCEPTED_TRANSLATION_STATUSES = {'ok', 'repaired_ok'}
RUSSIAN_STOPWORDS = {
    'и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты',
    'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему',
    'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь',
    'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней',
    'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж',
    'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем',
    'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть',
    'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя',
    'впрочем', 'хорошо', 'свою', 'этой', 'перед', 'иногда', 'лучше', 'чуть', 'том', 'нельзя', 'такой', 'им', 'более', 'всегда',
    'конечно', 'всю', 'между'
}


def normalize_label(label: str) -> str:
    return ' '.join(str(label).strip().lower().split())


def slugify_model_name(model_name: str) -> str:
    tail = model_name.rsplit('/', maxsplit=1)[-1]
    slug = re.sub(r'[^a-zA-Z0-9._-]+', '-', tail)
    return slug.strip('-').lower()


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False))
            handle.write('\n')


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def parse_json_records(path: Path) -> list[dict]:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        return []

    jsonl_rows = []
    jsonl_ok = True
    for line in text.splitlines():
        payload = line.strip()
        if not payload:
            continue
        try:
            parsed = json.loads(payload)
        except json.JSONDecodeError:
            jsonl_ok = False
            break
        if isinstance(parsed, list):
            jsonl_rows.extend(parsed)
        else:
            jsonl_rows.append(parsed)
    if jsonl_ok and jsonl_rows:
        return jsonl_rows

    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, dict):
            return [parsed]
    except json.JSONDecodeError:
        pass

    decoder = json.JSONDecoder()
    rows = []
    cursor = 0
    while cursor < len(text):
        while cursor < len(text) and text[cursor].isspace():
            cursor += 1
        if cursor >= len(text):
            break
        parsed, next_cursor = decoder.raw_decode(text, cursor)
        if isinstance(parsed, list):
            rows.extend(parsed)
        else:
            rows.append(parsed)
        cursor = next_cursor
        while cursor < len(text) and text[cursor] in ',\n\r\t ':
            cursor += 1

    if rows:
        return rows

    raise ValueError(f'Could not parse dataset file: {path}')


def iter_jsonl(path: Path) -> list[dict]:
    return parse_json_records(path)


def load_cocolofa_jsonl(path: Path) -> pd.DataFrame:
    rows = parse_json_records(path)

    df = pd.DataFrame(rows)
    assert 'label_str' in df.columns, 'label_str column is required.'
    assert 'sample_id' in df.columns, 'sample_id column is required.'
    assert 'split' in df.columns, 'split column is required.'
    assert 'text_ru' in df.columns, 'text_ru column is required.'

    df = df.copy()
    df['label_str'] = df['label_str'].map(normalize_label)
    df['label_id'] = pd.to_numeric(df.get('label_id'), errors='coerce').fillna(df['label_str'].map(LABEL_TO_ID)).astype(int)
    df['sample_id'] = pd.to_numeric(df['sample_id'], errors='coerce').astype(int)
    df['split'] = df['split'].astype(str).str.lower().str.strip()
    df['text_ru'] = df['text_ru'].fillna('').astype(str).str.strip()
    if 'text_masked' in df.columns:
        df['text_masked'] = df['text_masked'].fillna('').astype(str).str.strip()

    if 'translation_status' in df.columns:
        df = df[df['translation_status'].isin(ACCEPTED_TRANSLATION_STATUSES)]

    df = df[df['label_str'].isin(LABEL_TO_ID)]
    df = df[df['text_ru'].str.len() > 0]
    return df.reset_index(drop=True)


def get_fixed_splits(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df = df[df['split'] == 'train'].reset_index(drop=True)
    dev_df = df[df['split'] == 'dev'].reset_index(drop=True)
    test_df = df[df['split'] == 'test'].reset_index(drop=True)
    assert len(train_df) > 0 and len(dev_df) > 0 and len(test_df) > 0, 'Dataset must contain train/dev/test splits.'
    return train_df, dev_df, test_df


def tokenize_words(text: str) -> list[str]:
    return [token for token in re.findall(r'[A-Za-zА-Яа-яЁё]{2,}', str(text).lower()) if token not in RUSSIAN_STOPWORDS]


def top_words_by_class(df: pd.DataFrame, text_column: str, top_n: int = 15) -> pd.DataFrame:
    rows = []
    for label, group in df.groupby('label_str'):
        counter = Counter()
        for text in group[text_column].tolist():
            counter.update(tokenize_words(text))
        rows.append(
            {
                'label_str': label,
                'top_words': ', '.join(f'{word}:{count}' for word, count in counter.most_common(top_n)),
            }
        )
    return pd.DataFrame(rows).sort_values('label_str').reset_index(drop=True)


@dataclass
class BaselineConfig:
    dataset_path: str
    model_name: str
    text_column: str
    seed: int
    output_dir: str
    num_epochs: int = 5
    batch_size: int = 16
    eval_batch_size: int = 32
    learning_rate: float = 2e-5
    max_length: int = 256
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    patience: int = 2


class FallacyDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, text_column: str, max_length: int):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.text_column = text_column
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        row = self.df.iloc[idx]
        encoding = self.tokenizer(
            str(row[self.text_column]),
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(int(row['label_id']), dtype=torch.long),
            'sample_ids': torch.tensor(int(row['sample_id']), dtype=torch.long),
            'texts': str(row[self.text_column]),
        }


def compute_metrics(labels: list[int], preds: list[int], probabilities: np.ndarray | None = None) -> dict[str, Any]:
    label_order = list(range(N_CLASSES))
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, labels=label_order, average=None, zero_division=0)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)

    payload = {
        'accuracy': float(accuracy_score(labels, preds)),
        'macro_precision': float(macro_precision),
        'macro_recall': float(macro_recall),
        'macro_f1': float(macro_f1),
        'per_class_precision': {ID_TO_LABEL[idx]: float(value) for idx, value in zip(label_order, precision)},
        'per_class_recall': {ID_TO_LABEL[idx]: float(value) for idx, value in zip(label_order, recall)},
        'per_class_f1': {ID_TO_LABEL[idx]: float(value) for idx, value in zip(label_order, f1)},
        'classification_report': classification_report(
            labels,
            preds,
            labels=label_order,
            target_names=[ID_TO_LABEL[idx] for idx in label_order],
            output_dict=True,
            zero_division=0,
        ),
        'confusion_matrix': {
            'labels': [ID_TO_LABEL[idx] for idx in label_order],
            'matrix': confusion_matrix(labels, preds, labels=label_order).tolist(),
        },
    }

    if probabilities is not None:
        binarized = label_binarize(labels, classes=label_order)
        try:
            payload['roc_auc_macro_ovr'] = float(roc_auc_score(binarized, probabilities, multi_class='ovr', average='macro'))
            payload['roc_auc_weighted_ovr'] = float(roc_auc_score(binarized, probabilities, multi_class='ovr', average='weighted'))
        except ValueError:
            payload['roc_auc_macro_ovr'] = None
            payload['roc_auc_weighted_ovr'] = None

    return payload


def evaluate_model(model, loader, device, source_df: pd.DataFrame, text_column: str, dataset_path: Path, split_name: str):
    model.eval()
    preds: list[int] = []
    labels: list[int] = []
    probabilities: list[list[float]] = []
    prediction_rows: list[dict[str, Any]] = []

    with torch.no_grad():
        for batch in loader:
            sample_ids = batch['sample_ids'].cpu().tolist()
            gold_labels = batch['labels'].cpu().tolist()
            texts = list(batch['texts'])

            outputs = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
            )
            probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
            batch_preds = probs.argmax(axis=-1).tolist()

            preds.extend(batch_preds)
            labels.extend(gold_labels)
            probabilities.extend(probs.tolist())

            for sample_id, gold, pred, prob_vector, text in zip(sample_ids, gold_labels, batch_preds, probs.tolist(), texts):
                row = source_df[source_df['sample_id'] == int(sample_id)].iloc[0]
                prediction_rows.append(
                    {
                        'split': split_name,
                        'sample_id': int(sample_id),
                        'gold_label_id': int(gold),
                        'pred_label_id': int(pred),
                        'gold_label': ID_TO_LABEL[int(gold)],
                        'pred_label': ID_TO_LABEL[int(pred)],
                        'text_column': text_column,
                        'text': text,
                        'label_str': row['label_str'],
                        'source_dataset_path': str(dataset_path),
                        'probabilities': [float(value) for value in prob_vector],
                    }
                )

    metrics = compute_metrics(labels, preds, probabilities=np.asarray(probabilities))
    return metrics, prediction_rows


def train_one_run(config: BaselineConfig) -> dict[str, Any]:
    dataset_path = Path(config.dataset_path)
    output_dir = Path(config.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    set_global_seed(config.seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    logger.info('Training %s | %s | seed=%s | max_length=%s | device=%s', config.model_name, config.text_column, config.seed, config.max_length, device)

    df = load_cocolofa_jsonl(dataset_path)
    if config.text_column not in df.columns:
        raise KeyError(f'Missing text column: {config.text_column}')

    train_df, dev_df, test_df = get_fixed_splits(df)
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)

    train_dataset = FallacyDataset(train_df, tokenizer, config.text_column, config.max_length)
    dev_dataset = FallacyDataset(dev_df, tokenizer, config.text_column, config.max_length)
    test_dataset = FallacyDataset(test_df, tokenizer, config.text_column, config.max_length)

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=0)
    dev_loader = DataLoader(dev_dataset, batch_size=config.eval_batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=config.eval_batch_size, shuffle=False, num_workers=0)

    model = AutoModelForSequenceClassification.from_pretrained(
        config.model_name,
        num_labels=N_CLASSES,
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
        ignore_mismatched_sizes=True,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    total_training_steps = max(len(train_loader) * config.num_epochs, 1)
    warmup_steps = int(total_training_steps * config.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_training_steps)

    history = []
    best_macro_f1 = float('-inf')
    best_epoch = 0
    patience_counter = 0

    for epoch in range(1, config.num_epochs + 1):
        model.train()
        total_loss = 0.0
        step_count = 0

        for batch in train_loader:
            optimizer.zero_grad()
            outputs = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
                labels=batch['labels'].to(device),
            )
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += float(loss.item())
            step_count += 1

        dev_metrics, _ = evaluate_model(model, dev_loader, device, df, config.text_column, dataset_path, 'dev')
        epoch_summary = {
            'epoch': epoch,
            'train_loss': total_loss / max(step_count, 1),
            'val_macro_f1': dev_metrics['macro_f1'],
            'val_macro_precision': dev_metrics['macro_precision'],
            'val_macro_recall': dev_metrics['macro_recall'],
        }
        history.append(epoch_summary)
        logger.info('Epoch %s | train_loss=%.4f | val_macro_f1=%.4f', epoch, epoch_summary['train_loss'], epoch_summary['val_macro_f1'])

        if dev_metrics['macro_f1'] > best_macro_f1:
            best_macro_f1 = dev_metrics['macro_f1']
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), output_dir / 'best_model.pt')
            tokenizer.save_pretrained(output_dir / 'tokenizer')
        else:
            patience_counter += 1
            if patience_counter >= config.patience:
                logger.info('Early stopping at epoch %s', epoch)
                break

    model.load_state_dict(torch.load(output_dir / 'best_model.pt', map_location=device))
    dev_metrics, _ = evaluate_model(model, dev_loader, device, df, config.text_column, dataset_path, 'dev')
    test_metrics, test_predictions = evaluate_model(model, test_loader, device, df, config.text_column, dataset_path, 'test')

    config_payload = {
        **asdict(config),
        'dataset_path': str(dataset_path),
        'output_dir': str(output_dir),
        'model_slug': slugify_model_name(config.model_name),
        'device': str(device),
        'best_epoch': best_epoch,
    }
    dataset_summary = {
        'dataset_path': str(dataset_path),
        'accepted_row_count': int(len(df)),
        'train_count': int(len(train_df)),
        'val_count': int(len(dev_df)),
        'test_count': int(len(test_df)),
        'text_column': config.text_column,
    }

    write_json(output_dir / 'config.json', config_payload)
    write_json(output_dir / 'dataset_summary.json', dataset_summary)
    write_json(output_dir / 'training_history.json', history)
    write_json(output_dir / 'metrics_val.json', {key: value for key, value in dev_metrics.items() if key not in {'classification_report', 'confusion_matrix'}})
    write_json(output_dir / 'metrics_test.json', {key: value for key, value in test_metrics.items() if key not in {'classification_report', 'confusion_matrix'}})
    write_json(output_dir / 'classification_report_val.json', dev_metrics['classification_report'])
    write_json(output_dir / 'classification_report_test.json', test_metrics['classification_report'])
    write_json(output_dir / 'confusion_matrix_val.json', dev_metrics['confusion_matrix'])
    write_json(output_dir / 'confusion_matrix_test.json', test_metrics['confusion_matrix'])
    write_jsonl(output_dir / 'predictions_test.jsonl', test_predictions)

    return {
        'config': config_payload,
        'dataset_summary': dataset_summary,
        'metrics_test': test_metrics,
    }


def aggregate_run_directories(runs_root: Path, out_dir: Path) -> dict[str, Any]:
    run_dirs = sorted(path.parent for path in runs_root.rglob('metrics_test.json'))
    if not run_dirs:
        raise FileNotFoundError(f'No completed runs found under {runs_root}')

    groups: dict[str, list[dict[str, Any]]] = defaultdict(list)
    merged_predictions: list[dict[str, Any]] = []

    for run_dir in run_dirs:
        config = read_json(run_dir / 'config.json')
        metrics = read_json(run_dir / 'metrics_test.json')
        confusion = read_json(run_dir / 'confusion_matrix_test.json')
        predictions = iter_jsonl(run_dir / 'predictions_test.jsonl')

        group_key = f"{config['model_slug']}__{config['text_column']}__single"
        groups[group_key].append({'run_dir': str(run_dir), 'config': config, 'metrics': metrics, 'confusion': confusion})

        for prediction in predictions:
            merged_predictions.append(
                {
                    **prediction,
                    'run_dir': str(run_dir),
                    'model_slug': config['model_slug'],
                    'text_column': config['text_column'],
                    'seed': config['seed'],
                }
            )

    summary = {'runs_root': str(runs_root), 'run_count': len(run_dirs), 'groups': {}}
    confusion_payload = {}

    for group_key, runs in sorted(groups.items()):
        macro_f1 = [run['metrics']['macro_f1'] for run in runs]
        macro_precision = [run['metrics']['macro_precision'] for run in runs]
        macro_recall = [run['metrics']['macro_recall'] for run in runs]
        accuracy = [run['metrics']['accuracy'] for run in runs]
        roc_auc_macro = [run['metrics'].get('roc_auc_macro_ovr') for run in runs if run['metrics'].get('roc_auc_macro_ovr') is not None]

        per_class_values: dict[str, list[float]] = defaultdict(list)
        for run in runs:
            for label, value in run['metrics']['per_class_f1'].items():
                per_class_values[label].append(value)

        representative = runs[0]['config']
        summary['groups'][group_key] = {
            'model_name': representative['model_name'],
            'model_slug': representative['model_slug'],
            'text_column': representative['text_column'],
            'seed_count': len(runs),
            'macro_f1': {'mean': float(mean(macro_f1)), 'std': float(stdev(macro_f1)) if len(macro_f1) > 1 else 0.0},
            'macro_precision': {'mean': float(mean(macro_precision)), 'std': float(stdev(macro_precision)) if len(macro_precision) > 1 else 0.0},
            'macro_recall': {'mean': float(mean(macro_recall)), 'std': float(stdev(macro_recall)) if len(macro_recall) > 1 else 0.0},
            'accuracy': {'mean': float(mean(accuracy)), 'std': float(stdev(accuracy)) if len(accuracy) > 1 else 0.0},
            'roc_auc_macro_ovr': {
                'mean': float(mean(roc_auc_macro)) if roc_auc_macro else None,
                'std': float(stdev(roc_auc_macro)) if len(roc_auc_macro) > 1 else 0.0 if roc_auc_macro else None,
            },
            'per_class_f1': {
                label: {
                    'mean': float(mean(values)),
                    'std': float(stdev(values)) if len(values) > 1 else 0.0,
                }
                for label, values in sorted(per_class_values.items())
            },
            'run_dirs': [run['run_dir'] for run in runs],
        }
        confusion_payload[group_key] = runs[0]['confusion']

    out_dir.mkdir(parents=True, exist_ok=True)
    write_json(out_dir / 'summary_metrics.json', summary)
    write_json(out_dir / 'confusion_matrices.json', confusion_payload)
    write_jsonl(out_dir / 'merged_predictions.jsonl', merged_predictions)

    lines = [
        '| group | model | text | seeds | macro_f1 | macro_precision | macro_recall | roc_auc_macro_ovr |',
        '| --- | --- | --- | ---: | ---: | ---: | ---: | ---: |',
    ]
    for group_key, payload in summary['groups'].items():
        roc_auc_value = payload['roc_auc_macro_ovr']['mean']
        roc_auc_std = payload['roc_auc_macro_ovr']['std']
        roc_auc_text = 'n/a' if roc_auc_value is None else f"{roc_auc_value:.4f}±{roc_auc_std:.4f}"
        lines.append(
            '| {group} | {model} | {text} | {seeds} | {f1:.4f}±{f1_std:.4f} | {p:.4f}±{p_std:.4f} | {r:.4f}±{r_std:.4f} | {roc_auc} |'.format(
                group=group_key,
                model=payload['model_slug'],
                text=payload['text_column'],
                seeds=payload['seed_count'],
                f1=payload['macro_f1']['mean'],
                f1_std=payload['macro_f1']['std'],
                p=payload['macro_precision']['mean'],
                p_std=payload['macro_precision']['std'],
                r=payload['macro_recall']['mean'],
                r_std=payload['macro_recall']['std'],
                roc_auc=roc_auc_text,
            )
        )
    (out_dir / 'summary_table.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')

    return summary


def load_finetuned_run(run_dir: Path):
    config = read_json(run_dir / 'config.json')
    tokenizer = AutoTokenizer.from_pretrained(run_dir / 'tokenizer')
    model = AutoModelForSequenceClassification.from_pretrained(
        config['model_name'],
        num_labels=N_CLASSES,
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
        ignore_mismatched_sizes=True,
    )
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.load_state_dict(torch.load(run_dir / 'best_model.pt', map_location=device))
    model = model.to(device).eval()
    return config, tokenizer, model, device


def extract_cls_embeddings(model, tokenizer, texts: list[str], device, max_length: int = 256, batch_size: int = 32) -> np.ndarray:
    embeddings = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt',
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}
        with torch.no_grad():
            outputs = model(**encoded, output_hidden_states=True, return_dict=True)
            cls_embeddings = outputs.hidden_states[-1][:, 0, :].detach().cpu().numpy()
        embeddings.append(cls_embeddings)
    return np.concatenate(embeddings, axis=0)


In [ ]:
raw_df = load_cocolofa_jsonl(RAW_DATASET_PATH)
masked_df = load_cocolofa_jsonl(MASKED_DATASET_PATH)

print('raw rows:', len(raw_df))
print('masked rows:', len(masked_df))
print('raw split distribution:', dict(raw_df['split'].value_counts()))
print('masked split distribution:', dict(masked_df['split'].value_counts()))
assert len(raw_df) == len(masked_df), 'Raw and masked datasets must have the same row count.'
assert 'text_masked' in masked_df.columns, 'Masked dataset must contain text_masked.'

class_distribution = raw_df['label_str'].value_counts().sort_values(ascending=False)
plt.figure(figsize=(12, 5))
sns.barplot(x=class_distribution.index, y=class_distribution.values, color='#4472c4')
plt.xticks(rotation=45, ha='right')
plt.title('Class Distribution in COCOLOFA-RU v2')
plt.ylabel('Number of samples')
plt.xlabel('Label')
plt.tight_layout()
plt.show()

display(class_distribution.rename_axis('label_str').reset_index(name='count'))


In [ ]:
# Аналитика длины текстов по классам для raw и masked представлений.
analysis_tokenizer = AutoTokenizer.from_pretrained('DeepPavlov/rubert-base-cased')

raw_df = raw_df.copy()
masked_df = masked_df.copy()
raw_df['token_count'] = raw_df['text_ru'].apply(lambda text: len(analysis_tokenizer(text, truncation=False)['input_ids']))
masked_df['token_count'] = masked_df['text_masked'].apply(lambda text: len(analysis_tokenizer(text, truncation=False)['input_ids']))

length_summary = (
    raw_df.groupby('label_str')['token_count']
    .agg(['count', 'mean', 'median', 'std', 'max'])
    .round(2)
    .reset_index()
    .sort_values('mean', ascending=False)
)
display(length_summary)

length_plot_df = pd.concat(
    [
        raw_df[['label_str', 'token_count']].assign(representation='text_ru'),
        masked_df[['label_str', 'token_count']].assign(representation='text_masked'),
    ],
    ignore_index=True,
)

plt.figure(figsize=(15, 6))
sns.boxplot(data=length_plot_df, x='label_str', y='token_count', hue='representation', showfliers=False)
plt.xticks(rotation=45, ha='right')
plt.title('Token Count by Label and Text Representation')
plt.xlabel('Label')
plt.ylabel('Token count')
plt.tight_layout()
plt.show()


In [ ]:
# Самые частые слова по классам на raw русском тексте.
top_words_df = top_words_by_class(raw_df, text_column='text_ru', top_n=15)
display(top_words_df)


In [ ]:
BASELINE_MATRIX = [
    {
        'run_name': 'rubert_text_ru',
        'dataset_path': RAW_DATASET_PATH,
        'model_name': 'DeepPavlov/rubert-base-cased',
        'text_column': 'text_ru',
    },
    {
        'run_name': 'rubert_text_masked',
        'dataset_path': MASKED_DATASET_PATH,
        'model_name': 'DeepPavlov/rubert-base-cased',
        'text_column': 'text_masked',
    },
    {
        'run_name': 'xlmr_text_ru',
        'dataset_path': RAW_DATASET_PATH,
        'model_name': 'FacebookAI/xlm-roberta-base',
        'text_column': 'text_ru',
    },
    {
        'run_name': 'xlmr_text_masked',
        'dataset_path': MASKED_DATASET_PATH,
        'model_name': 'FacebookAI/xlm-roberta-base',
        'text_column': 'text_masked',
    },
]

planned_runs = []
for entry in BASELINE_MATRIX:
    for seed in SEEDS:
        planned_runs.append({**entry, 'seed': seed})

if MAX_RUNS is not None:
    planned_runs = planned_runs[:MAX_RUNS]

display(pd.DataFrame(planned_runs))


In [ ]:
completed_runs = []
failed_runs = []

for run in planned_runs:
    run_output_dir = OUTPUT_ROOT / run['run_name'] / f"seed_{run['seed']}"
    config = BaselineConfig(
        dataset_path=str(run['dataset_path']),
        model_name=run['model_name'],
        text_column=run['text_column'],
        seed=int(run['seed']),
        output_dir=str(run_output_dir),
        num_epochs=5,
        batch_size=16,
        eval_batch_size=32,
        learning_rate=2e-5,
        max_length=256,
        patience=2,
    )
    try:
        result = train_one_run(config)
        completed_runs.append(
            {
                'run_name': run['run_name'],
                'seed': run['seed'],
                'output_dir': str(run_output_dir),
                'macro_f1': result['metrics_test']['macro_f1'],
                'roc_auc_macro_ovr': result['metrics_test'].get('roc_auc_macro_ovr'),
            }
        )
    except Exception as exc:
        logger.exception('Run failed: %s | seed=%s', run['run_name'], run['seed'])
        failed_runs.append(
            {
                'run_name': run['run_name'],
                'seed': run['seed'],
                'output_dir': str(run_output_dir),
                'error': str(exc),
            }
        )

write_json(OUTPUT_ROOT / 'completed_runs.json', {'completed_runs': completed_runs, 'failed_runs': failed_runs})
display(pd.DataFrame(completed_runs).sort_values('macro_f1', ascending=False))
display(pd.DataFrame(failed_runs))


In [ ]:
assert completed_runs, 'No completed runs are available for aggregation.'
aggregate_payload = aggregate_run_directories(OUTPUT_ROOT, OUTPUT_ROOT / 'aggregate')
print((OUTPUT_ROOT / 'aggregate' / 'summary_table.md').read_text(encoding='utf-8'))

summary_df = pd.DataFrame(
    [
        {
            'group': group_key,
            'model_slug': payload['model_slug'],
            'text_column': payload['text_column'],
            'macro_f1_mean': payload['macro_f1']['mean'],
            'macro_f1_std': payload['macro_f1']['std'],
            'roc_auc_macro_mean': payload['roc_auc_macro_ovr']['mean'],
        }
        for group_key, payload in aggregate_payload['groups'].items()
    ]
).sort_values('macro_f1_mean', ascending=False)

plt.figure(figsize=(12, 5))
sns.barplot(data=summary_df, x='group', y='macro_f1_mean', color='#2e8b57')
plt.xticks(rotation=30, ha='right')
plt.title('Macro-F1 by Encoder Configuration')
plt.ylabel('Macro-F1 mean across seeds')
plt.xlabel('Configuration')
plt.tight_layout()
plt.show()

display(summary_df)


In [ ]:
# Конфузионная матрица и ROC-AUC для лучшего run по test macro-F1.
best_run = max(completed_runs, key=lambda row: row['macro_f1'])
BEST_RUN_DIR = Path(best_run['output_dir'])
best_config = read_json(BEST_RUN_DIR / 'config.json')
best_predictions = iter_jsonl(BEST_RUN_DIR / 'predictions_test.jsonl')

best_y_true = np.array([row['gold_label_id'] for row in best_predictions])
best_y_pred = np.array([row['pred_label_id'] for row in best_predictions])
best_y_score = np.array([row['probabilities'] for row in best_predictions])

cm = confusion_matrix(best_y_true, best_y_pred, labels=list(range(N_CLASSES)))
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[ID_TO_LABEL[i] for i in range(N_CLASSES)], yticklabels=[ID_TO_LABEL[i] for i in range(N_CLASSES)])
plt.title(f"Confusion Matrix | {best_config['model_slug']} | {best_config['text_column']} | seed={best_config['seed']}")
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
plt.show()

best_y_bin = label_binarize(best_y_true, classes=list(range(N_CLASSES)))
plt.figure(figsize=(10, 8))
for class_idx in range(N_CLASSES):
    if best_y_bin[:, class_idx].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(best_y_bin[:, class_idx], best_y_score[:, class_idx])
    class_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{ID_TO_LABEL[class_idx]} (AUC={class_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle='--', color='black')
plt.title('One-vs-Rest ROC Curves for Best Baseline Run')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

best_run


In [ ]:
# 2D и 3D проекция эмбеддингов лучшего encoder-run.
projection_df_source = load_cocolofa_jsonl(Path(best_config['dataset_path']))
text_column_for_projection = best_config['text_column']

projection_df = (
    projection_df_source.groupby('label_str', group_keys=False)
    .apply(lambda group: group.sample(n=min(len(group), 150), random_state=42))
    .reset_index(drop=True)
)

loaded_config, loaded_tokenizer, loaded_model, loaded_device = load_finetuned_run(BEST_RUN_DIR)
projection_embeddings = extract_cls_embeddings(
    loaded_model,
    loaded_tokenizer,
    projection_df[text_column_for_projection].tolist(),
    device=loaded_device,
    max_length=int(loaded_config['max_length']),
    batch_size=32,
)

pca_2d = PCA(n_components=2, random_state=42)
projection_2d = pca_2d.fit_transform(projection_embeddings)
plot_df_2d = projection_df[['sample_id', 'label_str']].copy()
plot_df_2d['pc1'] = projection_2d[:, 0]
plot_df_2d['pc2'] = projection_2d[:, 1]
fig_2d = px.scatter(plot_df_2d, x='pc1', y='pc2', color='label_str', hover_data=['sample_id'], title='PCA 2D Projection of Fine-Tuned Encoder Embeddings')
fig_2d.show()

pca_3d = PCA(n_components=3, random_state=42)
projection_3d = pca_3d.fit_transform(projection_embeddings)
plot_df_3d = projection_df[['sample_id', 'label_str']].copy()
plot_df_3d['pc1'] = projection_3d[:, 0]
plot_df_3d['pc2'] = projection_3d[:, 1]
plot_df_3d['pc3'] = projection_3d[:, 2]
fig_3d = px.scatter_3d(plot_df_3d, x='pc1', y='pc2', z='pc3', color='label_str', hover_data=['sample_id'], title='PCA 3D Projection of Fine-Tuned Encoder Embeddings')
fig_3d.show()


In [ ]:
# Зависимость качества от контекстного окна. По умолчанию выключено, потому что это дополнительные training-runs.
if RUN_CONTEXT_WINDOW_SWEEP:
    context_results = []
    context_windows = [64, 128, 256, 384]
    for max_length in context_windows:
        config = BaselineConfig(
            dataset_path=str(RAW_DATASET_PATH),
            model_name='DeepPavlov/rubert-base-cased',
            text_column='text_ru',
            seed=42,
            output_dir=str(OUTPUT_ROOT / 'context_window_sweep' / f'max_length_{max_length}'),
            num_epochs=5,
            batch_size=16,
            eval_batch_size=32,
            learning_rate=2e-5,
            max_length=max_length,
            patience=2,
        )
        result = train_one_run(config)
        context_results.append(
            {
                'max_length': max_length,
                'macro_f1': result['metrics_test']['macro_f1'],
                'roc_auc_macro_ovr': result['metrics_test'].get('roc_auc_macro_ovr'),
            }
        )

    context_df = pd.DataFrame(context_results).sort_values('max_length')
    display(context_df)
    plt.figure(figsize=(8, 4))
    sns.lineplot(data=context_df, x='max_length', y='macro_f1', marker='o')
    plt.title('Macro-F1 vs Context Window Length')
    plt.xlabel('max_length')
    plt.ylabel('Macro-F1')
    plt.tight_layout()
    plt.show()
else:
    print('Set RUN_CONTEXT_WINDOW_SWEEP = True if you want additional max_length experiments.')


In [ ]:
# Attention visualization for the best run using bertviz.
attention_source_df = load_cocolofa_jsonl(Path(best_config['dataset_path']))
misclassified_rows = [row for row in best_predictions if row['gold_label_id'] != row['pred_label_id']]
attention_row = misclassified_rows[0] if misclassified_rows else best_predictions[0]
attention_sample = attention_source_df[attention_source_df['sample_id'] == attention_row['sample_id']].iloc[0]
attention_text = str(attention_sample[best_config['text_column']])

print('sample_id:', int(attention_sample['sample_id']))
print('gold_label:', attention_row['gold_label'])
print('pred_label:', attention_row['pred_label'])
print('text_column:', best_config['text_column'])
print('text preview:', attention_text[:500])

encoded = loaded_tokenizer(attention_text, return_tensors='pt', truncation=True, max_length=int(best_config['max_length']))
encoded = {key: value.to(loaded_device) for key, value in encoded.items()}
with torch.no_grad():
    attention_outputs = loaded_model(**encoded, output_attentions=True, return_dict=True)
attention_tensors = [tensor.detach().cpu() for tensor in attention_outputs.attentions]
attention_tokens = loaded_tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
head_view(attention_tensors, attention_tokens)
